# Petrophysics with Non-Ideal Data

Not all well log data is suitable for standard petrophysical interpretation. This tutorial demonstrates:

- How to assess data quality before interpretation
- Recognizing when data doesn't fit standard models
- Working with scientific drilling data (ODP/IODP)
- Understanding the limitations of petrophysical equations

We'll use data from **ODP Site 1218A**, a scientific drilling site in the Pacific Ocean, which has very different characteristics from typical oil & gas wells.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import welly
from welly import Well
from welly.petro import (
    PetroInterpreter,
    PetrophysicalParameters,
    MatrixParameters,
    FluidParameters,
    ClayParameters,
)

print(f"welly version: {welly.__version__}")

## 1. Load the DLIS Data

This is data from Ocean Drilling Program Site 1218A in the equatorial Pacific.

In [ ]:
# Load from DLIS
well = Well.from_dlis('data/199-1218A_std-proc.dlis')

print(f"Well: {well.name}")
print(f"Curves: {len(well.data)}")
print(f"\nAvailable curves:")
for name in sorted(well.data.keys()):
    print(f"  {name}: {well.data[name].units}")

## 2. Initial Data Assessment

Before running any interpretation, let's look at the data characteristics.

In [ ]:
# Get depth
depth = well.data['HCGR'].df.index
print(f"Depth range: {depth.min():.1f} to {depth.max():.1f} m")
print(f"Note: Negative depths are above the seafloor (mudline)")

In [ ]:
# Examine key curves
curves_to_check = {
    'HCGR': 'Gamma Ray',
    'RHOB': 'Density',
    'APLC': 'Neutron Porosity',
    'ILD': 'Deep Resistivity',
}

print("Curve Statistics:")
print("=" * 60)
for mnem, desc in curves_to_check.items():
    if mnem in well.data:
        values = well.data[mnem].values
        valid = values[~np.isnan(values)]
        print(f"\n{desc} ({mnem}):")
        print(f"  Units: {well.data[mnem].units}")
        print(f"  Min: {valid.min():.2f}")
        print(f"  Max: {valid.max():.2f}")
        print(f"  Mean: {valid.mean():.2f}")
        print(f"  Std: {valid.std():.2f}")

## 3. Red Flags in the Data

Several things stand out as unusual compared to typical oil & gas wells:

In [ ]:
# Plot the data to visualize the issues
fig, axes = plt.subplots(1, 4, figsize=(14, 10), sharey=True)

# Gamma Ray - very low!
ax = axes[0]
gr = well.data['HCGR'].values
ax.plot(gr, depth, 'g-', lw=0.5)
ax.axvline(100, color='r', ls='--', label='Typical shale')
ax.set_xlabel('GR (gAPI)')
ax.set_ylabel('Depth (m)')
ax.set_xlim(0, 150)
ax.set_title('Gamma Ray\n⚠️ Very Low!')
ax.legend()

# Density - low
ax = axes[1]
rhob = well.data['RHOB'].values
ax.plot(rhob, depth, 'r-', lw=0.5)
ax.axvline(2.65, color='b', ls='--', label='Quartz matrix')
ax.set_xlabel('RHOB (g/cm3)')
ax.set_xlim(1.0, 3.0)
ax.set_title('Density\n⚠️ Low (unconsolidated)')
ax.legend()

# Neutron - very high!
ax = axes[2]
nphi = well.data['APLC'].values
ax.plot(nphi, depth, 'b-', lw=0.5)
ax.axvline(40, color='r', ls='--', label='Typical max')
ax.set_xlabel('APLC (%)')
ax.set_xlim(0, 150)
ax.set_title('Neutron Porosity\n⚠️ >100%!')
ax.legend()

# Resistivity
ax = axes[3]
rt = well.data['ILD'].values
ax.plot(rt, depth, 'k-', lw=0.5)
ax.set_xlabel('ILD (ohm.m)')
ax.set_xscale('log')
ax.set_xlim(0.1, 1000)
ax.set_title('Resistivity')

for ax in axes:
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)

plt.suptitle('ODP Site 1218A - Data Quality Issues', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Understanding the Issues

### Why is this data different?

1. **Very low gamma ray (0-20 gAPI)**
   - Typical shales: 100-150 gAPI
   - This is marine carbonate/siliceous ooze with very little clay
   - Standard Vshale equations won't work properly

2. **Low density (~1.5-2.0 g/cm3)**
   - Typical consolidated sandstone: 2.2-2.5 g/cm3
   - These are unconsolidated, water-saturated marine sediments
   - Very high porosity (40-60%)

3. **Neutron porosity >100%**
   - The APLC curve appears to be uncorrected or in unusual units
   - Cannot be used directly for porosity calculation

4. **Depth below seafloor**
   - Starts at -74m (above mudline) to 915m below seafloor
   - Not TVD or MD in the conventional sense

## 5. Attempting Standard Interpretation

Let's see what happens when we try to run a standard interpretation on this data.

In [ ]:
# Set up custom aliases for this dataset
custom_aliases = {
    'GR': ['HCGR', 'HSGR'],
    'RHOB': ['RHOB'],
    'NPHI': ['APLC'],  # This won't work well!
    'RT': ['ILD', 'IDPH'],
}

# Create interpreter
interp = PetroInterpreter(well, alias=custom_aliases)

# Check inputs
status = interp.check_inputs(verbose=True)

In [ ]:
# Try to calculate Vshale
# First, look at GR distribution
gr = well.data['HCGR'].values
valid_gr = gr[~np.isnan(gr)]

print("GR Statistics:")
print(f"  P5:  {np.percentile(valid_gr, 5):.1f} gAPI")
print(f"  P50: {np.percentile(valid_gr, 50):.1f} gAPI")
print(f"  P95: {np.percentile(valid_gr, 95):.1f} gAPI")
print(f"\nTypical oil/gas well:")
print(f"  Clean sand: 20-40 gAPI")
print(f"  Shale: 100-150 gAPI")
print(f"\n⚠️ This data has no shale baseline!")

In [ ]:
# Calculate Vshale anyway to demonstrate the problem
# Using the data's own range (which is wrong for standard interpretation)
params = PetrophysicalParameters(
    matrix=MatrixParameters.sandstone(),
    fluid=FluidParameters(rw=0.05),
    clay=ClayParameters(
        gr_clean=2,    # Using data's P5
        gr_shale=15,   # Using data's P95
    ),
    a=1.0, m=2.0, n=2.0,
    name='ODP Marine Sediments'
)

interp = PetroInterpreter(well, params=params, alias=custom_aliases)
vsh = interp.vshale(method='linear')

print(f"Vshale range: {np.nanmin(vsh.values):.2f} - {np.nanmax(vsh.values):.2f}")
print("\n⚠️ This Vshale is meaningless - it's just normalized GR, not actual shale content!")

In [ ]:
# Calculate density porosity
phi = interp.porosity(method='density', output='PHIT')

print(f"Porosity range: {np.nanmin(phi.values):.2f} - {np.nanmax(phi.values):.2f}")
print(f"Mean porosity: {np.nanmean(phi.values):.2f}")
print("\n✓ Density porosity is reasonable for unconsolidated marine sediments")
print("  (40-60% porosity is typical for shallow marine ooze)")

In [ ]:
# Calculate Sw
sw = interp.sw(method='archie', phi='PHIT')

print(f"Sw range: {np.nanmin(sw.values):.2f} - {np.nanmax(sw.values):.2f}")
print(f"Mean Sw: {np.nanmean(sw.values):.2f}")
print("\n✓ Sw ≈ 1.0 is correct - these are 100% water-saturated sediments")
print("  (No hydrocarbons expected in scientific drilling)")

## 6. Visualizing the Results

In [ ]:
# Plot the interpretation results
fig, axes = plt.subplots(1, 5, figsize=(16, 10), sharey=True)

# GR
ax = axes[0]
ax.plot(well.data['HCGR'].values, depth, 'g-', lw=0.5)
ax.set_xlabel('GR (gAPI)')
ax.set_ylabel('Depth (m)')
ax.set_xlim(0, 25)
ax.set_title('Gamma Ray')

# Vshale (with warning)
ax = axes[1]
ax.fill_betweenx(depth, 0, vsh.values, color='gray', alpha=0.5)
ax.set_xlabel('Vsh (v/v)')
ax.set_xlim(0, 1)
ax.set_title('Vshale\n⚠️ Not meaningful')

# Density
ax = axes[2]
ax.plot(well.data['RHOB'].values, depth, 'r-', lw=0.5)
ax.set_xlabel('RHOB (g/cm3)')
ax.set_xlim(1.0, 2.5)
ax.set_title('Density')

# Porosity
ax = axes[3]
ax.fill_betweenx(depth, 0, phi.values, color='blue', alpha=0.3)
ax.plot(phi.values, depth, 'b-', lw=0.5)
ax.set_xlabel('Porosity (v/v)')
ax.set_xlim(0, 0.8)
ax.set_title('Porosity\n✓ Reasonable')

# Sw
ax = axes[4]
ax.fill_betweenx(depth, 0, sw.values, color='lightblue', alpha=0.5)
ax.plot(sw.values, depth, 'b-', lw=0.5)
ax.set_xlabel('Sw (v/v)')
ax.set_xlim(0, 1.2)
ax.set_title('Sw\n✓ ~100% water')

for ax in axes:
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)

plt.suptitle('ODP Site 1218A - Interpretation Results (with caveats)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Lessons Learned

### When standard petrophysics doesn't apply:

1. **Check your GR range** - If GR is very low (<50 gAPI) or very high (>200 gAPI) throughout, standard Vshale equations may not apply.

2. **Check your density** - If RHOB is consistently <2.0 g/cm3, you may have unconsolidated sediments where standard matrix assumptions don't hold.

3. **Check your neutron** - If NPHI values are >50% or negative, the data may need correction or may be in non-standard units.

4. **Know your geology** - Marine carbonates, evaporites, coal, and volcanic rocks all require different interpretation approaches.

5. **Know your well type** - Scientific drilling (ODP/IODP), geothermal wells, and mining wells have different objectives and data characteristics.

### What worked here:

- **Density porosity** - The equation is physics-based and works for any lithology if you know the matrix density
- **Archie Sw** - Correctly shows ~100% water saturation (no hydrocarbons)

### What didn't work:

- **Vshale from GR** - No shale baseline in the data
- **Neutron porosity** - Data appears uncorrected or in wrong units
- **Net pay** - Meaningless for a scientific well with no hydrocarbons

## 8. Best Practices

Before running any petrophysical interpretation:

1. **Plot the raw data first** - Look for obvious issues
2. **Check curve statistics** - Are values in expected ranges?
3. **Understand the geology** - What lithologies are expected?
4. **Know the well type** - Exploration, development, scientific, geothermal?
5. **Validate with core data** - If available, calibrate to core measurements
6. **Use QC tools** - `interp.qc_inputs()` can help identify issues

In [ ]:
# Use the QC function
qc = interp.qc_inputs()

print("QC Results:")
print("=" * 50)
for curve, results in qc.items():
    if results['status'] != 'missing':
        print(f"\n{curve}:")
        print(f"  Status: {results['status']}")
        if results.get('warnings'):
            for w in results['warnings']:
                print(f"  ⚠️ {w}")

## Summary

This tutorial demonstrated that:

- Not all well log data is suitable for standard petrophysical interpretation
- Scientific drilling data (ODP/IODP) has different characteristics from oil & gas wells
- Always assess data quality before running interpretations
- Some calculations (density porosity, Archie Sw) are more robust than others
- Understanding the geology and well type is essential for meaningful interpretation